# Multi-task Reward Router Training (Utility + Task Type)

This notebook trains a multi-task VLM router that predicts:
1. **Task Type**: Classification head (which router task is this?)
2. **Utility-based Reward**: Regression head (predicting utility for `accuracy`, `cheap`, `fast`, `balanced` modes).

It loads data from a canonical parquet file, builds a multi-task dataset, trains the `MultiTaskRewardRouterModel`, and provides inference helpers.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import logging
import json
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Union

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Set project root to allow importing modules if needed
# Assuming notebook is in artemis_final/router_train/notebooks
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent
sys.path.append(str(PROJECT_ROOT))

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("multitask_router")

print(f"Project root set to: {PROJECT_ROOT}")
print(f"Torch version: {torch.__version__}")

In [ ]:
class RouterConfig:
    # Data
    data_path = "../../notebooks/data/router_profiles_with_utility.parquet"
    
    # Model
    text_encoder_name = "distilbert-base-uncased"
    max_seq_len = 256
    model_emb_dim = 32
    mode_emb_dim = 16
    hidden_dim = 256
    dropout = 0.1
    
    # Training
    batch_size = 256
    epochs = 10
    lr_encoder = 2e-5
    lr_head = 3e-4
    weight_decay = 0.01
    lambda_task = 0.3
    gradient_clip = 1.0
    seed = 42
    device = "cuda:1" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

cfg = RouterConfig()

# Set seeds
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(f"Device: {cfg.device}")

In [14]:
def load_and_prep_data(path):
    p = Path(path)
    if not p.exists():
        # Try searching for it
        p_alt = Path("../data/router_profiles_with_utility.parquet")
        if p_alt.exists():
            p = p_alt
        else:
            raise FileNotFoundError(f"Could not find data at {path} or {p_alt}")
    
    print(f"Loading data from {p}...")
    df = pd.read_parquet(p)
    
    # Filter valid rows
    # Required cols
    req_cols = ["sample_id", "router_task", "data_split", "prompt_text", "model_name", "ok"]
    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    # Filter ok=True
    len_orig = len(df)
    df = df[df["ok"] == True].copy()
    print(f"Filtered ok=True: {len_orig} -> {len(df)}")
    
    return df

profiles_df = load_and_prep_data(cfg.data_path)
display(profiles_df.head(2))

Filtered ok=True: 339056 -> 339056


,sample_id,run_id,source_config,source_dataset,source_index,router_task,ground_truth_type,data_split,prompt_text,ground_truth,...,total_tokens,utility_accuracy,utility_cheap,utility_fast,utility_balanced,cost_norm_new,lat_norm,glider_score,judge_molmo_score,judge_molmo_rank_group
0,finqa_700_03a1c065,run_20251208_023438,finqa,cauldron,700,table_math,numeric,train,( 1 ) adjusted other income ( expense ) exclud...,\nRationale: the average free cash flow provid...,...,1442,0.484211,0.674716,0.686095,0.729454,0.039526,0.011078,0.0,9.0,2.0
1,docvqa_20_62702d0d,run_20251207_225050,docvqa,cauldron,20,document_ocr,exact,train,What is the name of the company?\nEnsure brevi...,B&W.,...,4449,1.000000,0.951137,0.994905,0.966276,0.122159,0.012737,5.0,10.0,1.0


In [15]:
MODES = ["accuracy", "cheap", "fast", "balanced"]

def create_long_format(df, modes):
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building long-format"):
        base = {
            "sample_id": row["sample_id"],
            "router_task": row["router_task"],
            "data_split": row["data_split"],
            "prompt_text": row["prompt_text"],
            "model_name": row["model_name"],
            "prompt_len_words": row.get("txt_prompt_length_words", 0),
            "source_dataset": row.get("source_dataset", "unknown"),
        }
        
        # Add mode rows
        for mode in modes:
            # Check if utility_{mode} exists and is not null
            col_name = f"utility_{mode}"
            if col_name in row and pd.notnull(row[col_name]):
                r = base.copy()
                r["mode_name"] = mode
                r["utility_target"] = row[col_name]
                rows.append(r)
    
    return pd.DataFrame(rows)

df_long = create_long_format(profiles_df, MODES)
print(f"df_long shape: {df_long.shape}")
display(df_long.head())

Building long-format:   0%|          | 0/339056 [00:00<?, ?it/s]

df_long shape: (1356224, 9)


,sample_id,router_task,data_split,prompt_text,model_name,prompt_len_words,source_dataset,mode_name,utility_target
0,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,accuracy,0.484211
1,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,cheap,0.674716
2,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,fast,0.686095
3,finqa_700_03a1c065,table_math,train,( 1 ) adjusted other income ( expense ) exclud...,qwen2_5_vl_7b,970,cauldron,balanced,0.729454
4,docvqa_20_62702d0d,document_ocr,train,What is the name of the company?\nEnsure brevi...,qwen2_5_vl_7b,12,cauldron,accuracy,1.000000


In [16]:
# Create indices
model_names = sorted(df_long["model_name"].unique())
mode_names = MODES
task_names = sorted(df_long["router_task"].unique())

model_to_id = {m: i for i, m in enumerate(model_names)}
mode_to_id = {m: i for i, m in enumerate(mode_names)}
task_to_id = {t: i for i, t in enumerate(task_names)}

# Map to IDs
df_long["model_id"] = df_long["model_name"].map(model_to_id)
df_long["mode_id"] = df_long["mode_name"].map(mode_to_id)
df_long["task_id"] = df_long["router_task"].map(task_to_id)

# Save indices
os.makedirs("data", exist_ok=True)
with open("data/model_index.json", "w") as f: json.dump(model_names, f)
with open("data/mode_index.json", "w") as f: json.dump(mode_names, f)
with open("data/task_index.json", "w") as f: json.dump(task_names, f)

print("Indices saved to data/")
print(f"Models: {len(model_names)}")
print(f"Modes: {len(mode_names)}")
print(f"Tasks: {len(task_names)}")

Indices saved to data/
Models: 5
Modes: 4
Tasks: 30


In [ ]:
# Compute class weights for task classification
# Helps stabilize task prediction when classes are imbalanced

task_counts = df_long["task_id"].value_counts().sort_index()
total = task_counts.sum()
class_weights = total / (len(task_counts) * task_counts)
class_weights = torch.tensor(class_weights.values, dtype=torch.float32, device=cfg.device)

print("Task counts:", task_counts.to_dict())
print(f"Class weights: {class_weights}")


In [17]:
class MultiTaskRewardRouterDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Build prompt text
        # [ROUTER] Task: {router_task}. Source: {source_dataset}. Split: {data_split}. PromptLen: {len}. Question: {text}
        input_text = (
            f"[ROUTER] Task: {row['router_task']}. "
            f"Source: {row['source_dataset']}. "
            f"PromptLen: {row['prompt_len_words']}. "
            f"Question: {row['prompt_text']}"
        )
        
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            truncation=True,
            padding=False, # We handle padding in collate
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "model_id": row["model_id"],
            "mode_id": row["mode_id"],
            "utility_target": float(row["utility_target"]),
            "task_id": row["task_id"],
            "sample_id": row["sample_id"]
        }

def multitask_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    
    # Pad sequences
    max_len = max(len(x) for x in input_ids)
    input_ids_padded = []
    attention_mask_padded = []
    
    for ids, mask in zip(input_ids, attention_mask):
        pad_len = max_len - len(ids)
        input_ids_padded.append(ids + [0] * pad_len)
        attention_mask_padded.append(mask + [0] * pad_len)
        
    return {
        "input_ids": torch.tensor(input_ids_padded, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_padded, dtype=torch.long),
        "model_id": torch.tensor([x["model_id"] for x in batch], dtype=torch.long),
        "mode_id": torch.tensor([x["mode_id"] for x in batch], dtype=torch.long),
        "utility_target": torch.tensor([x["utility_target"] for x in batch], dtype=torch.float),
        "task_id": torch.tensor([x["task_id"] for x in batch], dtype=torch.long),
        "sample_ids": [x["sample_id"] for x in batch]
    }

In [18]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.text_encoder_name)

# Split data
train_df = df_long[df_long["data_split"] == "train"]
val_df = df_long[df_long["data_split"] == "val"]
test_df = df_long[df_long["data_split"] == "test"]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

train_ds = MultiTaskRewardRouterDataset(train_df, tokenizer, cfg.max_seq_len)
val_ds = MultiTaskRewardRouterDataset(val_df, tokenizer, cfg.max_seq_len)
test_ds = MultiTaskRewardRouterDataset(test_df, tokenizer, cfg.max_seq_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=multitask_collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=multitask_collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=multitask_collate_fn)

# Verify batch
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.shape}")

Train: 951912, Val: 204452, Test: 199860
Batch shapes:
input_ids: torch.Size([256, 256])
attention_mask: torch.Size([256, 256])
model_id: torch.Size([256])
mode_id: torch.Size([256])
utility_target: torch.Size([256])
task_id: torch.Size([256])


In [19]:
class MultiTaskRewardRouterModel(nn.Module):
    def __init__(self, config, num_models, num_modes, num_tasks):
        super().__init__()
        self.config = config
        
        # Text Encoder
        self.text_encoder = AutoModel.from_pretrained(config.text_encoder_name)
        text_hidden_size = self.text_encoder.config.hidden_size
        
        # Embeddings
        self.model_embedding = nn.Embedding(num_models, config.model_emb_dim)
        self.mode_embedding = nn.Embedding(num_modes, config.mode_emb_dim)
        
        # Routing Head
        # Input: [text, model, mode]
        input_dim = text_hidden_size + config.model_emb_dim + config.mode_emb_dim
        self.routing_mlp = nn.Sequential(
            nn.Linear(input_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(config.hidden_dim // 2, 1) # scalar utility
        )
        
        # Task Head
        # Input: [text] only
        self.task_head = nn.Sequential(
            nn.Linear(text_hidden_size, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, num_tasks)
        )
        
    def forward(self, input_ids, attention_mask, model_id, mode_id):
        # Encode text
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use CLS token (index 0)
        h_text = outputs.last_hidden_state[:, 0, :]
        
        # Task Prediction (Text only)
        task_logits = self.task_head(h_text)
        
        # Utility Prediction
        h_model = self.model_embedding(model_id)
        h_mode = self.mode_embedding(mode_id)
        
        h_combined = torch.cat([h_text, h_model, h_mode], dim=-1)
        utility_hat = self.routing_mlp(h_combined).squeeze(-1)
        
        return {
            "utility_hat": utility_hat,
            "task_logits": task_logits
        }

model = MultiTaskRewardRouterModel(
    cfg, 
    num_models=len(model_names),
    num_modes=len(mode_names),
    num_tasks=len(task_names)
)
model.to(cfg.device)
print("Model initialized.")

Model initialized.


In [ ]:
num_training_steps = len(train_loader) * cfg.epochs

optimizer = torch.optim.AdamW([
    {"params": model.text_encoder.parameters(), "lr": cfg.lr_encoder},
    {"params": model.routing_mlp.parameters(), "lr": cfg.lr_head},
    {"params": model.task_head.parameters(), "lr": cfg.lr_head},
    {"params": model.model_embedding.parameters(), "lr": cfg.lr_head},
    {"params": model.mode_embedding.parameters(), "lr": cfg.lr_head},
], weight_decay=cfg.weight_decay)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps)

criterion_utility = nn.MSELoss()
criterion_task = nn.CrossEntropyLoss(weight=class_weights)


class RewardRouterTrainer:
    def __init__(
        self,
        model,
        optimizer,
        scheduler,
        train_loader,
        val_loader,
        criterion_utility,
        criterion_task,
        device,
        lambda_task=0.3,
        gradient_clip=1.0,
        epochs=10,
        patience=3,
        ckpt_path="best_multitask_router.pt",
    ):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion_utility = criterion_utility
        self.criterion_task = criterion_task
        self.device = device
        self.lambda_task = lambda_task
        self.gradient_clip = gradient_clip
        self.epochs = epochs
        self.patience = patience
        self.ckpt_path = ckpt_path

    def _to_device(self, batch):
        return {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    def _safe_corr(self, preds, targets):
        if len(preds) < 2:
            return 0.0
        try:
            corr = pearsonr(preds, targets)[0]
            if np.isnan(corr):
                return 0.0
            return float(corr)
        except Exception:
            return 0.0

    def train(self):
        history = []
        best_val_loss = float("inf")
        patience_counter = 0

        for epoch_idx in range(self.epochs):
            train_metrics = self.train_epoch()
            val_metrics = self.validate()

            epoch_log = {
                "epoch": epoch_idx + 1,
                **train_metrics,
                **val_metrics,
            }
            history.append(epoch_log)

            print(
                f"Epoch {epoch_idx + 1}: "
                f"train_loss={train_metrics['train_loss']:.4f} | "
                f"val_loss={val_metrics['val_loss']:.4f} | "
                f"train_task_acc={train_metrics['train_task_acc']:.4f} | "
                f"val_task_acc={val_metrics['val_task_acc']:.4f} | "
                f"train_corr={train_metrics['train_corr']:.4f} | "
                f"val_corr={val_metrics['val_corr']:.4f}"
            )

            if val_metrics["val_loss"] < best_val_loss:
                best_val_loss = val_metrics["val_loss"]
                torch.save(self.model.state_dict(), self.ckpt_path)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping triggered.")
                    break

        return history


class MultiTaskTrainer(RewardRouterTrainer):
    def train_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_loss_routing = 0.0
        total_loss_task = 0.0
        task_correct = 0
        task_total = 0

        preds_u = []
        targets_u = []

        for batch in tqdm(self.train_loader, desc="Train epoch"):
            batch = self._to_device(batch)
            self.optimizer.zero_grad()

            out = self.model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                model_id=batch["model_id"],
                mode_id=batch["mode_id"],
            )

            loss_routing = self.criterion_utility(out["utility_hat"], batch["utility_target"])
            loss_task = self.criterion_task(out["task_logits"], batch["task_id"])
            loss = loss_routing + self.lambda_task * loss_task

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
            self.optimizer.step()
            if self.scheduler is not None:
                self.scheduler.step()

            total_loss += loss.item()
            total_loss_routing += loss_routing.item()
            total_loss_task += loss_task.item()

            preds = torch.argmax(out["task_logits"], dim=1)
            task_correct += (preds == batch["task_id"]).sum().item()
            task_total += batch["task_id"].numel()

            preds_u.extend(out["utility_hat"].detach().cpu().tolist())
            targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        avg_loss = total_loss / max(len(self.train_loader), 1)
        avg_loss_routing = total_loss_routing / max(len(self.train_loader), 1)
        avg_loss_task = total_loss_task / max(len(self.train_loader), 1)
        task_acc = task_correct / max(task_total, 1)
        train_corr = self._safe_corr(preds_u, targets_u)

        return {
            "train_loss": avg_loss,
            "train_loss_routing": avg_loss_routing,
            "train_loss_task": avg_loss_task,
            "train_task_acc": task_acc,
            "train_corr": train_corr,
        }

    def validate(self):
        self.model.eval()
        total_loss = 0.0
        total_loss_routing = 0.0
        total_loss_task = 0.0
        task_correct = 0
        task_total = 0

        preds_u = []
        targets_u = []

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Val epoch"):
                batch = self._to_device(batch)

                out = self.model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    model_id=batch["model_id"],
                    mode_id=batch["mode_id"],
                )

                loss_routing = self.criterion_utility(out["utility_hat"], batch["utility_target"])
                loss_task = self.criterion_task(out["task_logits"], batch["task_id"])
                loss = loss_routing + self.lambda_task * loss_task

                total_loss += loss.item()
                total_loss_routing += loss_routing.item()
                total_loss_task += loss_task.item()

                preds = torch.argmax(out["task_logits"], dim=1)
                task_correct += (preds == batch["task_id"]).sum().item()
                task_total += batch["task_id"].numel()

                preds_u.extend(out["utility_hat"].detach().cpu().tolist())
                targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        avg_loss = total_loss / max(len(self.val_loader), 1)
        avg_loss_routing = total_loss_routing / max(len(self.val_loader), 1)
        avg_loss_task = total_loss_task / max(len(self.val_loader), 1)
        task_acc = task_correct / max(task_total, 1)
        val_corr = self._safe_corr(preds_u, targets_u)

        return {
            "val_loss": avg_loss,
            "val_loss_routing": avg_loss_routing,
            "val_loss_task": avg_loss_task,
            "val_task_acc": task_acc,
            "val_corr": val_corr,
        }


trainer = MultiTaskTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_utility=criterion_utility,
    criterion_task=criterion_task,
    device=cfg.device,
    lambda_task=cfg.lambda_task,
    gradient_clip=cfg.gradient_clip,
    epochs=cfg.epochs,
    patience=3,
    ckpt_path="best_multitask_router.pt",
)

history = trainer.train()


## 10.1 Training Curves


In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

# 1) Loss curves
plt.figure()
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# 2) Task accuracy
plt.figure()
plt.plot(history_df["epoch"], history_df["train_task_acc"], label="train_task_acc")
plt.plot(history_df["epoch"], history_df["val_task_acc"], label="val_task_acc")
plt.xlabel("Epoch")
plt.ylabel("Task Accuracy")
plt.title("Task Classification Accuracy")
plt.legend()
plt.show()

# 3) Correlation
plt.figure()
plt.plot(history_df["epoch"], history_df["train_corr"], label="train_corr")
plt.plot(history_df["epoch"], history_df["val_corr"], label="val_corr")
plt.xlabel("Epoch")
plt.ylabel("Pearson Corr (Utility)")
plt.title("Utility Correlation vs Epoch")
plt.legend()
plt.show()


## 11. Evaluation: Routing Accuracy vs Oracle


In [ ]:
ckpt_path = "best_multitask_router.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=cfg.device))
model.eval()
print(f"Loaded best model from {ckpt_path}")

# 11.1 Task classification accuracy on test set
test_task_correct = 0
test_task_total = 0

test_preds_u = []
test_targets_u = []
eval_records = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Eval"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"],
        )

        utility_hat = outputs["utility_hat"]
        task_logits = outputs["task_logits"]
        task_pred = task_logits.argmax(dim=-1)

        test_task_correct += (task_pred == batch["task_id"]).sum().item()
        test_task_total += batch["task_id"].numel()

        test_preds_u.extend(utility_hat.detach().cpu().tolist())
        test_targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        for i in range(len(batch["sample_ids"])):
            eval_records.append({
                "sample_id": batch["sample_ids"][i],
                "mode_name": mode_names[int(batch["mode_id"][i].cpu())],
                "model_name": model_names[int(batch["model_id"][i].cpu())],
                "utility_target": float(batch["utility_target"][i].cpu()),
                "utility_pred": float(utility_hat[i].cpu()),
                "task_id": int(batch["task_id"][i].cpu()),
                "task_name": task_names[int(batch["task_id"][i].cpu())],
                "task_pred_id": int(task_pred[i].cpu()),
                "task_pred_name": task_names[int(task_pred[i].cpu())],
            })

test_task_acc = test_task_correct / max(test_task_total, 1)
try:
    test_corr = float(pearsonr(test_preds_u, test_targets_u)[0])
except Exception:
    test_corr = 0.0

print(f"Test task classification accuracy: {test_task_acc:.4f}")
print(f"Test utility correlation: {test_corr:.4f}")

eval_df = pd.DataFrame(eval_records)
display(eval_df.head())


In [ ]:
print("Evaluating Routing Accuracy vs Oracle on test set...")
routing_metrics_per_mode = []

for mode in mode_names:
    mode_df = eval_df[eval_df["mode_name"] == mode]

    total_samples = 0
    routing_hits = 0
    oracle_utility_sum = 0.0
    router_utility_sum = 0.0

    for sid, group in mode_df.groupby("sample_id"):
        if len(group) < 2:
            continue

        oracle_row = group.loc[group["utility_target"].idxmax()]
        pred_row = group.loc[group["utility_pred"].idxmax()]

        total_samples += 1
        if pred_row["model_name"] == oracle_row["model_name"]:
            routing_hits += 1

        oracle_utility_sum += oracle_row["utility_target"]
        router_utility_sum += pred_row["utility_target"]

    if total_samples > 0:
        oracle_mean = oracle_utility_sum / total_samples
        router_mean = router_utility_sum / total_samples
        gap = oracle_mean - router_mean
        recovery = router_mean / oracle_mean if oracle_mean > 0 else 0.0

        routing_metrics_per_mode.append({
            "mode": mode,
            "routing_accuracy": routing_hits / total_samples,
            "oracle_utility_mean": oracle_mean,
            "router_utility_mean": router_mean,
            "utility_gap": gap,
            "recovery": recovery,
            "test_task_acc": test_task_acc,
            "test_utility_corr": test_corr,
        })

summary = pd.DataFrame(routing_metrics_per_mode)
display(summary)

if not summary.empty:
    ax = summary.set_index("mode")[
        ["routing_accuracy", "recovery"]
    ].plot(kind="bar", figsize=(8, 4), ylim=(0, 1.05), title="Routing Accuracy and Recovery (Test)")
    ax.set_ylabel("Score")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 12. Save Artifacts


In [ ]:
os.makedirs("results", exist_ok=True)

if 'eval_df' in globals():
    eval_df.to_csv("results/multitask_eval_details.csv", index=False)
    print("Saved detailed eval to results/multitask_eval_details.csv")
if 'summary' in globals():
    summary.to_csv("results/multitask_eval_summary.csv", index=False)
    print("Saved summary to results/multitask_eval_summary.csv")


## 13. Inference Demo: Routing + Task Prediction


In [ ]:
id_to_task = {v: k for k, v in task_to_id.items()}

def route_query(
    model,
    tokenizer,
    prompt_text: str,
    mode_name: str,
    model_names: List[str],
    model_to_id: Dict[str, int],
    mode_to_id: Dict[str, int],
    id_to_task: Dict[int, str],
    device: str = "cuda",
    temperature: float = 1.0,
    task_top_k: int = 3,
):
    model.eval()

    input_text = (
        f"[ROUTER] Task: unknown. "
        f"Source: inference. "
        f"PromptLen: {len(prompt_text.split())}. "
        f"Question: {prompt_text}"
    )

    encoding = tokenizer(
        input_text,
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    num_cands = len(model_names)
    input_ids = input_ids.repeat(num_cands, 1)
    attention_mask = attention_mask.repeat(num_cands, 1)

    model_ids = torch.tensor([model_to_id[m] for m in model_names], device=device)
    mode_id_val = mode_to_id.get(mode_name, 0)
    mode_ids = torch.tensor([mode_id_val] * num_cands, device=device)

    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            model_id=model_ids,
            mode_id=mode_ids,
        )

        utility_scores = out["utility_hat"]
        task_logits = out["task_logits"][0]

    # Model routing
    scores = utility_scores / temperature
    probs = torch.softmax(scores, dim=0)

    scores_dict = {m: float(s) for m, s in zip(model_names, utility_scores)}
    probs_dict = {m: float(p) for m, p in zip(model_names, probs)}

    # Task prediction
    task_probs = torch.softmax(task_logits, dim=0)
    task_pred_idx = int(task_probs.argmax().item())
    task_pred_name = id_to_task[task_pred_idx]
    task_conf = float(task_probs[task_pred_idx])

    k = min(task_top_k, len(task_probs))
    topk_probs, topk_indices = torch.topk(task_probs, k=k)
    task_topk = [
        {"task": id_to_task[int(idx)], "prob": float(p)}
        for p, idx in zip(topk_probs, topk_indices)
    ]

    task_probs_dict = {
        id_to_task[i]: float(task_probs[i])
        for i in range(len(task_probs))
    }

    return {
        "scores": scores_dict,
        "probs": probs_dict,
        "task_probs": task_probs_dict,
        "task_type_pred": task_pred_name,
        "task_confidence": task_conf,
        "task_topk": task_topk,
    }


In [ ]:
from pprint import pprint

sid = test_df["sample_id"].sample(1, random_state=cfg.seed).iloc[0]
sample = test_df[test_df["sample_id"] == sid].iloc[0]
prompt_text = sample["prompt_text"]

mode_name = "accuracy"
model_candidates = list(model_to_id.keys())

out = route_query(
    model=model,
    tokenizer=tokenizer,
    prompt_text=prompt_text,
    mode_name=mode_name,
    model_names=model_candidates,
    model_to_id=model_to_id,
    mode_to_id=mode_to_id,
    id_to_task=id_to_task,
    device=cfg.device,
    temperature=1.0,
    task_top_k=3,
)

print("PROMPT:
", prompt_text[:500], "...
")
print("Predicted task:", out["task_type_pred"], f"(conf={out['task_confidence']:.2f})")
print("
Top-3 tasks:")
pprint(out["task_topk"])
print("
Model scores:")
pprint(out["scores"])
print("
Model probs:")
pprint(out["probs"])
